In [1]:
# Version for filename
ver = 'GRU'  # GRU or LSTM

date = "250804"
rnn_folder = f"D:/Python_TK_3/datas/{date}_DL"

shap_number = "VisStim_evoked_01"
#os.mkdir(f"{rnn_folder}/shap{shap_number}")

In [2]:
# データのサンプリングレート
fs = 20  # サンプリング周波数
first_ex = 0
last_ex = 39
NumberOfDatas = last_ex - first_ex + 1        # number of experiments
start_stim = 20
stop_stim = 30
calc_start = 5
calc_end = 39
start_ave= 5,
end_ave= 11
look_frame = 10   

In [3]:
import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np

In [4]:
date = "250804"
mixed_dataset = "ss-visStim_evoked_01"

testX  = np.load(f"{rnn_folder}/model_{mixed_dataset}/{date}_{mixed_dataset}_test_features.npy")
testY  = np.load(f"{rnn_folder}/model_{mixed_dataset}/{date}_{mixed_dataset}_test_targets.npy")

print(testX.shape, testY.shape)

(384, 11, 128, 135, 3) (384,)


In [5]:
# すでに testX/testY はロード済みとする
model = tf.keras.models.load_model(
    f"{rnn_folder}/model_{shap_number}/cnn_model_5epoch.h5",
    compile=False
)

# 取り出したい層名
target_layer_name = "td_flatten"

# レイヤ存在チェック（名前が違う場合に備えて可視化）
try:
    target_layer = model.get_layer(target_layer_name)
except ValueError:
    print("利用可能なレイヤ名一覧：", [l.name for l in model.layers])
    raise

# ★ここがポイント：既存モデルの入力を使う
cnn_extractor = tf.keras.Model(
    inputs=model.inputs,                    # または model.input
    outputs=target_layer.output
)

# 推論（バッチサイズは必要に応じて調整）
test_features = cnn_extractor.predict(testX, batch_size=32, verbose=1)
print(test_features.shape)   # 期待: (n_samples, frames, features)

c:\Python\anaconda3\envs\cnn\Lib\site-packages\keras\src\models\functional.py:225: UserWarning: The structure of `inputs` doesn't match the expected structure: ['input_frames']. Received: the structure of inputs=*
  warnings.warn(


12/12 ━━━━━━━━━━━━━━━━━━━━ 3s 194ms/step
(384, 11, 67584)


In [6]:
np.save(f"{rnn_folder}/model_{shap_number}/{date}_{shap_number}_cnn_detailed_test_features.npy", test_features)